# Vehicle Routing Problem in Pyomo

Lecture: *Week 10 - Vehicle Routing Problem*, the slides (`1 Lecture/Vehicle Routing Problem.pdf`) and the notes (`1 Lecture/Notes/Routing Problems - Lecture Notes.md`). The last slide links to a Google Colab notebook; this notebook does the same job and runs in the browser.

**The problem.** A depot (node 1) serves 7 customers, each with a demand, and every vehicle can carry at most `capacity`. The notebook builds the three models from the lecture on this one small example, solves each with the HiGHS solver and draws the routes:

1. **TSP with MTZ constraints** (one vehicle, no capacity), with and without the subtour constraints
2. **VRP, minimise total distance**: several vehicles with a capacity; a flow variable carries the load
3. **VRP, minimise the longest route**: each vehicle has its own variables (the "deliver as quickly
   as possible" objective)

**How to run it.** Run the cells in order. The first cell installs Pyomo and HiGHS (allow a minute the first time); the rest takes about 15 seconds in the browser, most of it for model 3. Change the data cell and re-run to explore.

**Expected results.** TSP without MTZ: length **233.4**, made of four two-city loops; with MTZ: one tour 1 → 8 → 2 → 3 → 4 → 5 → 6 → 7 → 1, length **281.4**. VRP, total distance: **327.6**, with two full vehicles (load 15 each; routes of length 172.8 and 154.8). VRP, longest route: **137.4**, with all three vehicles (114.2, 110.9 and 137.4; total 362.5). A route may be printed in the opposite direction; its length is the same.

In [ ]:
# Run this cell first (installs the extra packages; safe to re-run).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

## The data and helper functions

Node 1 is the depot and nodes 2–8 are customers, each with (x, y) coordinates and a demand; the distance between two nodes is the straight-line distance, rounded to one decimal. The helper functions read a solution: `arcs_used` lists the arcs whose binary variable is 1, `routes_from_arcs` follows them out of the depot to recover each route, `route_length` adds up a route's distances and `plot_arcs` draws the arcs, one colour per vehicle.

In [ ]:
import math
import matplotlib.pyplot as plt
import pyomo.environ as pyo

# depot = node 1, customers = 2..8
coords = {1: (50, 50), 2: (20, 80), 3: (35, 90), 4: (80, 85), 5: (90, 60),
          6: (75, 20), 7: (40, 15), 8: (15, 40)}
demand = {1: 0, 2: 4, 3: 3, 4: 5, 5: 4, 6: 6, 7: 3, 8: 5}
capacity = 15          # load one vehicle can carry
num_vehicles = 3

N = list(coords)
n = len(N)
# straight-line distance for every arc i -> j (no arc from a node to itself)
dist = {(i, j): round(math.dist(coords[i], coords[j]), 1) for i in N for j in N if i != j}
print(f"{n - 1} customers, total demand {sum(demand.values())}, "
      f"{num_vehicles} vehicles x capacity {capacity} = {num_vehicles * capacity}")

# HiGHS through Pyomo's APPSI interface: it runs in the browser (GLPK and CBC do not)
solver = pyo.SolverFactory("appsi_highs")

def arcs_used(x, k=None):
    """List the arcs (i, j) whose binary variable is 1."""
    # > 0.5 rather than == 1: the solver may return 0.9999999 for a chosen arc
    if k is None:
        return [(i, j) for (i, j) in dist if x[i, j].value > 0.5]
    return [(i, j) for (i, j) in dist if x[i, j, k].value > 0.5]

def routes_from_arcs(arcs):
    """Follow arcs out of the depot to recover each route as a list of nodes."""
    nxt = {}
    for i, j in arcs:
        nxt.setdefault(i, []).append(j)
    routes = []
    for start in nxt.get(1, []):
        route, node = [1, start], start
        while node != 1:
            node = nxt[node][0]
            route.append(node)
        routes.append(route)
    return routes

def route_length(route):
    return sum(dist[a, b] for a, b in zip(route, route[1:]))

def plot_arcs(arc_groups, title):
    fig, ax = plt.subplots(figsize=(5, 5))
    for idx, arcs in enumerate(arc_groups):
        colour = f"C{idx}"
        for i, j in arcs:
            (x0, y0), (x1, y1) = coords[i], coords[j]
            ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle="->", color=colour, lw=1.8))
    for i, (x, y) in coords.items():
        ax.scatter(x, y, s=250 if i == 1 else 120, color="black" if i == 1 else "white",
                   edgecolor="black", zorder=3)
        ax.text(x, y, str(i), ha="center", va="center", color="white" if i == 1 else "black",
                fontsize=9, zorder=4)
    ax.set_title(title); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_aspect("equal")
    plt.show()

## 1. TSP with the Miller–Tucker–Zemlin (MTZ) constraints

x[i, j] = 1 if the tour goes from i to j; u[i] orders the nodes along the tour (only the order of the values matters,
not the values themselves). Each node has one arc in and one arc out. The cell solves the model twice: first with the
MTZ constraints switched off (`deactivate()`), when the solver returns several small **subtours**, then with them
switched on, which leaves one tour.

In [ ]:
tsp = pyo.ConcreteModel()
tsp.x = pyo.Var(list(dist), within=pyo.Binary)      # x[i, j] = 1 if the tour uses arc i -> j
tsp.u = pyo.Var(N[1:], bounds=(1, n - 1))           # order of the nodes; node 1 (the start) has no u
# objective: total length of the arcs used
tsp.obj = pyo.Objective(expr=sum(dist[a] * tsp.x[a] for a in dist), sense=pyo.minimize)
# every node is left exactly once
tsp.out_one = pyo.Constraint(N, rule=lambda m, i: sum(m.x[i, j] for j in N if j != i) == 1)
# every node is entered exactly once
tsp.in_one = pyo.Constraint(N, rule=lambda m, i: sum(m.x[j, i] for j in N if j != i) == 1)
# MTZ, for i and j other than node 1: if x[i, j] = 1 the right-hand side is 0, so u[j] >= u[i] + 1;
# if x[i, j] = 0 it is n - 1 and the constraint restricts nothing
tsp.mtz = pyo.Constraint([(i, j) for (i, j) in dist if i != 1 and j != 1],
                         rule=lambda m, i, j: m.u[i] - m.u[j] + 1 <= (n - 1) * (1 - m.x[i, j]))

tsp.mtz.deactivate()
solver.solve(tsp)
print(f"Without MTZ: length {pyo.value(tsp.obj):.1f}")
plot_arcs([arcs_used(tsp.x)], "Without MTZ: subtours appear")

tsp.mtz.activate()
solver.solve(tsp)
tour = routes_from_arcs(arcs_used(tsp.x))[0]
print(f"With MTZ: length {pyo.value(tsp.obj):.1f}, tour {' -> '.join(map(str, tour))}")
plot_arcs([arcs_used(tsp.x)], "With MTZ: one tour")

## 2. VRP: minimise total distance

Customers still have one arc in and one out, but up to `num_vehicles` arcs may leave the depot.
The flow f[i, j] is the load on the vehicle as it travels from i to j: each customer keeps its demand
(flow in − flow out = demand), and f[i, j] ≤ capacity × x[i, j]. This flow also rules out subtours, which
is why the lecture's model has no MTZ constraint ("Why didn't we include the sub-tour constraints?").

In [ ]:
vrp = pyo.ConcreteModel()
vrp.x = pyo.Var(list(dist), within=pyo.Binary)            # x[i, j] = 1 if some vehicle drives i -> j
vrp.f = pyo.Var(list(dist), within=pyo.NonNegativeReals)  # f[i, j] = load it carries on that arc
# objective: total distance driven by all vehicles
vrp.obj = pyo.Objective(expr=sum(dist[a] * vrp.x[a] for a in dist), sense=pyo.minimize)

C = N[1:]   # customers
# every customer is left once and entered once (not the depot: several vehicles use it)
vrp.out_one = pyo.Constraint(C, rule=lambda m, i: sum(m.x[i, j] for j in N if j != i) == 1)
vrp.in_one = pyo.Constraint(C, rule=lambda m, i: sum(m.x[j, i] for j in N if j != i) == 1)
# at most num_vehicles routes start at the depot
vrp.leave_depot = pyo.Constraint(expr=sum(vrp.x[1, j] for j in C) <= num_vehicles)
# flow conservation: the load drops by demand[i] at customer i (this also rules out subtours)
vrp.flow = pyo.Constraint(C, rule=lambda m, i:
    sum(m.f[j, i] for j in N if j != i) - sum(m.f[i, j] for j in N if j != i) == demand[i])
# load can travel only on a used arc, and never above the capacity
vrp.cap = pyo.Constraint(list(dist), rule=lambda m, i, j: m.f[i, j] <= capacity * m.x[i, j])

solver.solve(vrp)
routes = routes_from_arcs(arcs_used(vrp.x))
print(f"Total distance {pyo.value(vrp.obj):.1f}")
for r in routes:
    print(f"  {' -> '.join(map(str, r))}: load {sum(demand[i] for i in r)}, length {route_length(r):.1f}")
plot_arcs([list(zip(r, r[1:])) for r in routes], "VRP: minimise total distance")

## 3. VRP: minimise the longest route

Now each vehicle k has its own arc and flow variables, and Cmax is at least the length of every
vehicle's route. Minimising Cmax balances the work: with the data above the longest route drops from 172.8 to 137.4,
while the total distance rises from 327.6 to 362.5.

The constraint `mm.cont` (a vehicle that enters a node also leaves it) is needed: without it vehicle k could arrive at
a customer and another vehicle leave, and the route lengths would mean nothing. The *Full Model* slide lists it under
"Optional"; it belongs in the model.

In [ ]:
K = list(range(1, num_vehicles + 1))
A = list(dist)
mm = pyo.ConcreteModel()
mm.x = pyo.Var(A, K, within=pyo.Binary)
mm.f = pyo.Var(A, K, within=pyo.NonNegativeReals)
mm.Cmax = pyo.Var(within=pyo.NonNegativeReals)
mm.obj = pyo.Objective(expr=mm.Cmax, sense=pyo.minimize)

# every customer is entered once, by exactly one vehicle
mm.visit = pyo.Constraint(C, rule=lambda m, i: sum(m.x[j, i, k] for j in N if j != i for k in K) == 1)
# a vehicle that enters a node also leaves it
mm.cont = pyo.Constraint(N, K, rule=lambda m, i, k:
    sum(m.x[j, i, k] for j in N if j != i) == sum(m.x[i, j, k] for j in N if j != i))
# each vehicle leaves the depot at most once
mm.depot = pyo.Constraint(K, rule=lambda m, k: sum(m.x[1, j, k] for j in C) <= 1)
# load flow per vehicle (also removes subtours)
mm.flow = pyo.Constraint(C, K, rule=lambda m, i, k:
    sum(m.f[j, i, k] for j in N if j != i) - sum(m.f[i, j, k] for j in N if j != i)
    == demand[i] * sum(m.x[j, i, k] for j in N if j != i))
mm.cap = pyo.Constraint(A, K, rule=lambda m, i, j, k: m.f[i, j, k] <= capacity * m.x[i, j, k])
# Cmax is at least each vehicle's route length
mm.longest = pyo.Constraint(K, rule=lambda m, k: sum(dist[a] * m.x[a, k] for a in A) <= m.Cmax)

solver.solve(mm)
groups = []
print(f"Longest route {pyo.value(mm.Cmax):.1f}")
for k in K:
    arcs = arcs_used(mm.x, k)
    if arcs:
        r = routes_from_arcs(arcs)[0]
        groups.append(arcs)
        print(f"  vehicle {k}: {' -> '.join(map(str, r))}, load {sum(demand[i] for i in r)}, "
              f"length {route_length(r):.1f}")
print(f"Total distance {sum(route_length(routes_from_arcs(g)[0]) for g in groups):.1f}")
plot_arcs(groups, "VRP: minimise the longest route")

## What do you notice?

Compare the plots of models 2 and 3. Model 2 uses only two of the three vehicles. Why does minimising the total
distance leave a vehicle at the depot, and what changes when the longest route is minimised?

<details><summary>Answer</summary>

Every extra route has to drive out of the depot and back again, so a few long routes are shorter in total than many
short ones. Model 2 fills two vehicles to exactly 15 (the total demand is 30 = 2 × 15), and its longest route is 172.8.
Model 3 ignores the total: it spreads the customers over all three vehicles (loads 9, 9 and 12), so the longest route
falls to 137.4 while the total rises to 362.5. This is the "long routes for some vans, short for others" issue on the
slides.
</details>

## Try it
- With `capacity = 15`, model 2 uses only two of the three vehicles. Set `capacity = 12` and re-run: what happens to the routes?
- Set `num_vehicles = 2` and `capacity = 14`: why is there no solution?
- Add a customer 9 at (60, 70) with demand 2.
- Compare the total distance of model 2 and model 3: what does balancing the routes cost?

<details><summary>Answers</summary>

- **`capacity = 12`.** Two vehicles now carry only 24, less than the total demand 30, so model 2 must use all three:
  total distance 362.5, routes 1 → 4 → 5 → 1, 1 → 6 → 7 → 1 and 1 → 8 → 2 → 3 → 1 (loads 9, 9 and 12; a route may
  print reversed). Models 1 and 3 do not change (281.4 and 137.4): model 3's routes already carry at most 12.
- **`num_vehicles = 2`, `capacity = 14`.** The fleet carries at most 2 × 14 = 28, less than the total demand 30 (the
  first line printed by the data cell shows both), so no set of routes is feasible. The model-2 cell stops with
  `RuntimeError: A feasible solution was not found…`: that error is the answer. Set the data back and re-run.
- **Customer 9.** Add `9: (60, 70)` to `coords` and `9: 2` to `demand`. The TSP tour grows to 288.8. The total demand
  is now 32, more than two vehicles can carry, so model 2 uses three routes (total 363.8). Model 3's longest route
  stays 137.4: customer 9 joins the route 1 → 9 → 4 → 5 → 1 (115.5).
- **Model 2 vs model 3.** Balancing costs 362.5 − 327.6 = 34.9 more in total (+10.7 %) and makes the longest route
  35.4 shorter (172.8 → 137.4, −20.5 %): later deliveries finish sooner, at the price of more driving.
</details>